# Data Exploration
Initial exploration of CT scan and tabular metadata.


In [ ]:
import pylidc as pl
import os

# Tell pylidc where your data is
os.environ['LIDC_IDRI_PATH'] = r'C:\Users\User\Desktop\Lung Cancer Detection\data\raw\LIDC-IDRI'

# Check how many scans are available
scans = pl.query(pl.Scan).all()
print(f"Total scans found: {len(scans)}")

In [14]:
from pathlib import Path
import configparser
import pylidc.Scan as pl_scan

# Load the first scan
scan = scans[0]

# Print basic information about it
print(f"Patient ID:        {scan.patient_id}")
print(f"Slice thickness:   {scan.slice_thickness} mm")
print(f"Pixel spacing:     {scan.pixel_spacing} mm")
print(f"Number of slices:  {scan.slice_zvals.shape[0]}")

# Python 3.13 compatibility shim for pylidc using deprecated SafeConfigParser
if not hasattr(configparser, "SafeConfigParser"):
    configparser.SafeConfigParser = configparser.ConfigParser

# Point pylidc to this project's local LIDC-IDRI directory
candidate_paths = [
    Path.cwd() / "data" / "raw" / "LIDC-IDRI",
    Path.cwd().parent / "data" / "raw" / "LIDC-IDRI",
]
dicom_root = next((p for p in candidate_paths if p.exists()), None)
if dicom_root is None:
    raise FileNotFoundError("Could not find data/raw/LIDC-IDRI from current notebook location.")

pl_scan._get_dicom_file_path_from_config_file = lambda: str(dicom_root)

# Load the actual 3D volume as a numpy array
volume = scan.to_volume()
print(f"\nCT scan shape:     {volume.shape}")
print(f"Min HU value:      {volume.min()}")
print(f"Max HU value:      {volume.max()}")

Patient ID:        LIDC-IDRI-0078
Slice thickness:   3.0 mm
Pixel spacing:     0.65 mm
Number of slices:  87
Loading dicom files ... This may take a moment.

CT scan shape:     (512, 512, 87)
Min HU value:      -2048
Max HU value:      8155


In [ ]:
# TODO:import os
import numpy as np
import matplotlib.pyplot as plt
import pylidc as pl

print("All libraries imported successfully!") 


In [ ]:
# Show 3 slices from the CT scan
# top, middle, and bottom of the chest

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Top slice
axes[0].imshow(volume[:, :, 20], cmap='gray')
axes[0].set_title('Top slice (slice 20)')
axes[0].axis('off')

# Middle slice
axes[1].imshow(volume[:, :, volume.shape[2]//2], cmap='gray')
axes[1].set_title('Middle slice')
axes[1].axis('off')

# Bottom slice
axes[2].imshow(volume[:, :, 70], cmap='gray')
axes[2].set_title('Bottom slice (slice 70)')
axes[2].axis('off')

plt.suptitle(f'CT Scan: {scan.patient_id}', fontsize=14)
plt.tight_layout()
plt.show()